In [ ]:
import numpy as np
import pandas as pd

# ---------- Normal pdf/cdf (pure NumPy) ----------
def norm_pdf(x):
    x = np.asarray(x, dtype=float)
    return np.exp(-0.5 * x * x) / np.sqrt(2.0 * np.pi)

def norm_cdf(x):
    x = np.asarray(x, dtype=float)
    sign = np.sign(x)
    z = np.abs(x) / np.sqrt(2.0)
    t = 1.0 / (1.0 + 0.3275911 * z)
    a1,a2,a3,a4,a5 = 0.254829592, -0.284496736, 1.421413741, -1.453152027, 1.061405429
    erf_approx = 1.0 - (((((a5*t + a4)*t + a3)*t + a2)*t + a1)*t) * np.exp(-z*z)
    erf_approx = sign * erf_approx
    return 0.5 * (1.0 + erf_approx)

# ---------- Load & preprocess & small-sample extraction ----------
def load_sample_and_prepare(csv_path, n_per_maturity=100, max_maturities=10, r=0.02, q=0.0, random_state=42):
    """
    Required columns: date, exdate, cp_flag, strike_price, best_bid, best_offer, (optional) impl_volatility.
    Automatically: compute T (years); rescale abnormally large strikes per OPRA convention (>5e4 ⇒ divide by 1000);
    estimate, for each maturity, the spot S via a near-ATM put–call parity and obtain S_use (fallback to K_true if estimation fails).
    Then: randomly sample n_per_maturity rows for each maturity.
    """
    df = pd.read_csv(csv_path)
    df['date_dt']   = pd.to_datetime(df['date'])
    df['exdate_dt'] = pd.to_datetime(df['exdate'])
    # Prevent 0 days to expiry: at least 1 day
    df['T'] = (df['exdate_dt'] - df['date_dt']).dt.days.clip(lower=1) / 365.0
    df['mid'] = (df['best_bid'] + df['best_offer']) / 2.0

    # Strike scaling (OPRA: index options often *1000)
    medK = df['strike_price'].median()
    scale = 1000.0 if medK > 5e4 else 1.0
    df['K_true'] = df['strike_price'] / scale

    # For each maturity, estimate S via near-ATM put–call parity: S = (C-P)e^{qT} + K e^{-rT}
    est_S = {}
    for exd, g in df.groupby('exdate'):
        g = g.copy()
        calls = g[g['cp_flag'].str.upper()=='C'][['K_true','mid','T']]
        puts  = g[g['cp_flag'].str.upper()=='P'][['K_true','mid','T']]
        m = pd.merge(calls, puts, on=['K_true','T'], how='inner', suffixes=('_C','_P'))
        if len(m)==0:
            est_S[exd] = np.nan
            continue
        m['diff'] = np.abs(m['mid_C'] - m['mid_P'])  # pick near-ATM
        row = m.loc[m['diff'].idxmin()]
        K = float(row['K_true']); C = float(row['mid_C']); P = float(row['mid_P']); T = float(row['T'])
        S_hat = (C - P) * np.exp(q*T) + K * np.exp(-r*T)
        est_S[exd] = S_hat

    df['S_est'] = df['exdate'].map(est_S)
    df['S_use'] = np.where(np.isfinite(df['S_est']), df['S_est'], df['K_true'])

    # Keep the first few maturities and sample within groups
    mats = sorted(df['exdate'].unique())[:max_maturities]
    df = df[df['exdate'].isin(mats)]
    sample = (df.groupby('exdate', group_keys=False)
                .apply(lambda g: g.sample(min(n_per_maturity, len(g)), random_state=random_state))
                ).reset_index(drop=True)
    return sample

# ---------- Black–Scholes (Price/Delta/Theta/Vega/Rho) ----------
def bs_vectorized_price_delta_theta_vega_rho(df, r=0.02, q=0.0):
    S = df['S_use'].to_numpy(float)
    K = df['K_true'].to_numpy(float)
    T = df['T'].to_numpy(float)
    cp = df['cp_flag'].astype(str).str.upper().to_numpy()
    sigma = df.get('impl_volatility', pd.Series(np.nan, index=df.index)).to_numpy(float)
    sigma = np.where(np.isfinite(sigma) & (sigma>1e-6), sigma, 0.15)

    sqrtT = np.sqrt(T)
    d1 = (np.log(S/K) + (r - q + 0.5*sigma**2)*T) / (sigma*sqrtT)
    d2 = d1 - sigma*sqrtT
    Nd1, Nd2, nd1 = norm_cdf(d1), norm_cdf(d2), norm_pdf(d1)

    # Price
    call = S*np.exp(-q*T)*Nd1 - K*np.exp(-r*T)*Nd2
    put  = call - S*np.exp(-q*T) + K*np.exp(-r*T)
    price = np.where(cp=='C', call, put)

    # Delta
    delta = np.where(cp=='C', np.exp(-q*T)*Nd1, np.exp(-q*T)*(Nd1-1.0))

    # Theta (annualized)
    theta_call = -(S*np.exp(-q*T)*nd1*sigma)/(2*sqrtT) - r*K*np.exp(-r*T)*Nd2 + q*S*np.exp(-q*T)*Nd1
    theta_put  = -(S*np.exp(-q*T)*nd1*sigma)/(2*sqrtT) + r*K*np.exp(-r*T)*norm_cdf(-d2) - q*S*np.exp(-q*T)*norm_cdf(-d1)
    theta = np.where(cp=='C', theta_call, theta_put)

    # Vega (w.r.t. σ)
    vega = S*np.exp(-q*T) * nd1 * sqrtT

    # Rho (w.r.t. r)
    rho_call =  K * T * np.exp(-r*T) * Nd2
    rho_put  = -K * T * np.exp(-r*T) * norm_cdf(-d2)
    rho = np.where(cp=='C', rho_call, rho_put)

    return pd.DataFrame({
        'price_bs': price,
        'delta_bs': delta,
        'theta_bs': theta,
        'vega_bs':  vega,
        'rho_bs':   rho
    })

# ---------- PDE: Crank–Nicolson + Thomas (Price/Delta/Theta) ----------
def cn_price_delta_theta_fast(S0,K,T,r,sigma,cp,Smax_mult=3.0,M=120,N=120, theta_method='timefd'):
    def thomas(a,b,c,d):
        n=len(b); c=c.astype(float).copy(); d=d.astype(float).copy(); b=b.astype(float).copy()
        for i in range(1,n):
            w=a[i-1]/b[i-1]; b[i]-=w*c[i-1]; d[i]-=w*d[i-1]
        x=np.empty(n); x[-1]=d[-1]/b[-1]
        for i in range(n-2,-1,-1): x[i]=(d[i]-c[i]*x[i+1])/b[i]
        return x

    Smax=max(Smax_mult*K,1.5*S0); dS=Smax/M; N=max(N,1); dt=T/N
    S=np.linspace(0.0,Smax,M+1)
    V=np.maximum(S-K,0.0) if cp=='C' else np.maximum(K-S,0.0)

    i=np.arange(1,M); Si=S[i]
    A=0.5*sigma**2*Si**2/dS**2 - 0.5*r*Si/dS
    C=0.5*sigma**2*Si**2/dS**2 + 0.5*r*Si/dS
    B=-(A+C)-r

    # Rannacher: two half-steps
    h=0.5*dt; aI=-h*A[1:]; bI=1-h*B; cI=-h*C[:-1]
    for _ in range(2):
        rhs=V[1:M].copy(); tau=h
        V0=0.0 if cp=='C' else K*np.exp(-r*tau)
        VN=(Smax-K*np.exp(-r*tau)) if cp=='C' else 0.0
        rhs[0]-=(-h*A[0])*V0; rhs[-1]-=(-h*C[-1])*VN
        V[1:M]=thomas(aI,bI,cI,rhs); V[0],V[-1]=V0,VN

    aL=-0.5*dt*A[1:]; bL=1-0.5*dt*B; cL=-0.5*dt*C[:-1]
    aR= 0.5*dt*A[1:]; bR=1+0.5*dt*B; cR= 0.5*dt*C[:-1]

    V_at_dt = None
    for n in range(N):
        rhs=bR*V[1:M]; rhs[1:]+=aR*V[1:M-1]; rhs[:-1]+=cR*V[2:M]
        tn=n*dt; tn1=(n+1)*dt
        if cp=='C':
            V0n=0.0; VNn=Smax-K*np.exp(-r*tn)
            V0n1=0.0; VNn1=Smax-K*np.exp(-r*tn1)
        else:
            V0n=K*np.exp(-r*tn);   VNn=0.0
            V0n1=K*np.exp(-r*tn1); VNn1=0.0
        rhs[0]+=0.5*dt*(A[0]*V0n  - A[0]*V0n1)
        rhs[-1]+=0.5*dt*(C[-1]*VNn - C[-1]*VNn1)

        if n == N-2:
            V_tmp = thomas(aL,bL,cL,rhs)
            V_at_dt = np.empty(M+1); V_at_dt[1:M] = V_tmp; V_at_dt[0],V_at_dt[-1] = V0n1,VNn1

        V[1:M]=thomas(aL,bL,cL,rhs); V[0],V[-1]=V0n1,VNn1

    idx=int(np.clip(int(round(S0/dS)),1,M-1))
    price=float(np.interp(S0,S,V))
    Delta=float((V[idx+1]-V[idx-1])/(2*dS))

    if theta_method=='timefd' and V_at_dt is not None:
        price_dt=float(np.interp(S0,S,V_at_dt))
        Theta=(price_dt - price)/dt
    else:
        VS=Delta; VSS=(V[idx+1]-2*V[idx]+V[idx-1])/(dS**2)
        Theta=-(0.5*sigma**2*S[idx]**2*VSS + r*S[idx]*VS - r*V[idx])

    return price, Delta, float(Theta)

# ---------- PDE: Vega/Rho via bump central differences ----------
def pde_vega_rho_bump(S0,K,T,r,sigma,cp,
                      Smax_mult=3.0,M=120,N=120,
                      dv=1e-3, dr=1e-4):
    p_up  = cn_price_delta_theta_fast(S0,K,T,r,sigma+dv,cp,Smax_mult,M,N)[0]
    p_dn  = cn_price_delta_theta_fast(S0,K,T,r,sigma-dv,cp,Smax_mult,M,N)[0]
    vega  = (p_up - p_dn) / (2.0*dv)

    p_upr = cn_price_delta_theta_fast(S0,K,T,r+dr,sigma,cp,Smax_mult,M,N)[0]
    p_dnr = cn_price_delta_theta_fast(S0,K,T,r-dr,sigma,cp,Smax_mult,M,N)[0]
    rho   = (p_upr - p_dnr) / (2.0*dr)
    return float(vega), float(rho)

# ---------- Randomly pick 5 rows: BS vs PDE (including all Greeks) ----------
def compare_bs_vs_pde_on_five_with_greeks(csv_path,
                                          r=0.02, q=0.0,
                                          M=400, N=600, Smax_mult=4.0,
                                          dv=1e-3, dr=1e-4,
                                          random_state=7):
    # Sampling pool → randomly pick 5 rows
    pool = load_sample_and_prepare(csv_path,
                                   n_per_maturity=1000, max_maturities=999,
                                   r=r, q=q, random_state=random_state)
    pool = pool.dropna(subset=['best_bid','best_offer'])
    samp5 = pool.sample(min(5, len(pool)), random_state=random_state).reset_index(drop=True)

    # BS closed-form (including vega/rho)
    bs = bs_vectorized_price_delta_theta_vega_rho(samp5, r=r, q=q)

    # PDE: price/Delta/Theta + (bump) Vega/Rho
    pde_price, pde_delta, pde_theta, pde_vega, pde_rho = [], [], [], [], []
    for _, row in samp5.iterrows():
        S0=float(row['S_use']); K=float(row['K_true']); T=float(row['T'])
        sigma=row.get('impl_volatility', np.nan)
        sigma=float(sigma) if np.isfinite(sigma) and sigma>1e-8 else 0.15
        cp=str(row['cp_flag']).upper()

        p,d,th = cn_price_delta_theta_fast(S0,K,T,r,sigma,cp,
                                           Smax_mult=Smax_mult, M=M, N=N,
                                           theta_method='timefd')
        pde_price.append(p); pde_delta.append(d); pde_theta.append(th)

        vega, rho = pde_vega_rho_bump(S0,K,T,r,sigma,cp,
                                      Smax_mult=Smax_mult, M=M, N=N,
                                      dv=dv, dr=dr)
        pde_vega.append(vega); pde_rho.append(rho)

    # Aggregate
    out = samp5.copy()
    out[['price_bs','delta_bs','theta_bs','vega_bs','rho_bs']] = bs[
        ['price_bs','delta_bs','theta_bs','vega_bs','rho_bs']
    ]
    out['price_pde'] = pde_price
    out['delta_pde'] = pde_delta
    out['theta_pde'] = pde_theta
    out['vega_pde']  = pde_vega
    out['rho_pde']   = pde_rho

    # Errors
    out['err_price'] = out['price_pde'] - out['price_bs']
    out['err_delta'] = out['delta_pde'] - out['delta_bs']
    out['err_theta'] = out['theta_pde'] - out['theta_bs']
    out['err_vega']  = out['vega_pde']  - out['vega_bs']
    out['err_rho']   = out['rho_pde']   - out['rho_bs']
    out = add_relative_errors_percent(out)

    # Brief summary
    def mae(x):  return float(np.nanmean(np.abs(x)))
    def rmse(x): return float(np.sqrt(np.nanmean(np.square(x))))
    def mape_pct(col): return float(np.nanmean(out[col]))  # already absolute percentages

    summary = pd.Series({
        'MAE_price': mae(out['err_price']),
        'RMSE_price': rmse(out['err_price']),
        'MAE_delta': mae(out['err_delta']),
        'RMSE_delta': rmse(out['err_delta']),
        'MAE_theta': mae(out['err_theta']),
        'RMSE_theta': rmse(out['err_theta']),
        'MAE_vega':  mae(out['err_vega']),
        'RMSE_vega': rmse(out['err_vega']),
        'MAE_rho':   mae(out['err_rho']),
        'RMSE_rho':  rmse(out['err_rho']),
        'MAPE_price_%': mape_pct('abs_relerr_price_pct'),
        'MAPE_delta_%': mape_pct('abs_relerr_delta_pct'),
        'MAPE_theta_%': mape_pct('abs_relerr_theta_pct'),
        'MAPE_vega_%':  mape_pct('abs_relerr_vega_pct'),
        'MAPE_rho_%':   mape_pct('abs_relerr_rho_pct'),
        'rows': len(out)
    })

    cols = ['date','exdate','cp_flag','K_true','T','impl_volatility',
            'price_bs','price_pde','err_price',
            'delta_bs','delta_pde','err_delta',
            'theta_bs','theta_pde','err_theta',
            'vega_bs','vega_pde','err_vega',
            'rho_bs','rho_pde','err_rho']
    out = out[cols].sort_values(['exdate','K_true']).reset_index(drop=True)
    return out, summary

# —— Utility: add relative error (%) vs BS baseline, and its absolute value ——
def add_relative_errors_percent(out,
                                eps_price=1e-8, eps_delta=1e-8,
                                eps_theta=1e-6, eps_vega=1e-8, eps_rho=1e-8):
    """
    Add new columns to DataFrame out:
    relerr_*_pct     = 100 * err_* / max(|*_bs|, eps)
    abs_relerr_*_pct = |relerr_*_pct|
    This makes quantities with different scales comparable and prevents blow-ups when denominators are close to zero.
    """

    def _relpct(err_col, bs_col, eps):
        base = out[bs_col].abs().clip(lower=eps)
        rel  = 100.0 * out[err_col] / base
        return rel, rel.abs()

    # Price
    out['relerr_price_pct'], out['abs_relerr_price_pct'] = _relpct('err_price', 'price_bs', eps_price)
    # Delta (in [-1,1]; use eps when near 0 to avoid blow-up)
    out['relerr_delta_pct'], out['abs_relerr_delta_pct'] = _relpct('err_delta', 'delta_bs', eps_delta)
    # Theta (often larger and can be ±; use a slightly larger floor)
    out['relerr_theta_pct'], out['abs_relerr_theta_pct'] = _relpct('err_theta', 'theta_bs', eps_theta)
    # Vega / Rho
    if 'err_vega' in out:
        out['relerr_vega_pct'], out['abs_relerr_vega_pct'] = _relpct('err_vega', 'vega_bs', eps_vega)
    if 'err_rho' in out:
        out['relerr_rho_pct'], out['abs_relerr_rho_pct']   = _relpct('err_rho',  'rho_bs',  eps_rho)

    return out


In [24]:
out5, summary5 = compare_bs_vs_pde_on_five_with_greeks(
    "SPX_Aug.csv",
    r=0.02, q=0.0,
    M=400, N=600, Smax_mult=4.0,
    dv=1e-3, dr=1e-4,
    random_state=7
)
print(summary5)
out5

/tmp/ipykernel_1694/3143974762.py:62: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(n_per_maturity, len(g)), random_state=random_state))


MAE_price        0.066737
RMSE_price       0.082209
MAE_delta        0.004896
RMSE_delta       0.007039
MAE_theta        5.222291
RMSE_theta      10.030180
MAE_vega         3.146523
RMSE_vega        4.656065
MAE_rho          0.704664
RMSE_rho         0.909914
MAPE_price_%     0.468112
MAPE_delta_%     2.921481
MAPE_theta_%     1.300324
MAPE_vega_%      2.599053
MAPE_rho_%       0.735089
rows             5.000000
dtype: float64


,date,exdate,cp_flag,K_true,T,impl_volatility,price_bs,price_pde,err_price,delta_bs,...,err_delta,theta_bs,theta_pde,err_theta,vega_bs,vega_pde,err_vega,rho_bs,rho_pde,err_rho
0,2023-08-09,2023-08-25,C,4030.0,0.043836,0.212185,557.646088,557.667759,0.021671,0.998366,...,0.000055,-92.628768,-93.517233,-0.888465,5.061948,5.465231,0.403283,176.170224,176.406141,0.235917
1,2023-08-23,2023-08-29,P,4520.0,0.016438,0.144636,90.555837,90.482695,-0.073142,-0.841271,...,-0.013554,-529.026943,-506.709121,22.317823,137.623836,132.693585,-4.930250,-62.817362,-62.599796,0.217566
2,2023-08-31,2023-09-12,P,4120.0,0.032877,0.233029,4.761118,4.867975,0.106857,-0.058784,...,0.007403,-325.795189,-324.310082,1.485107,93.413926,93.241154,-0.172771,-8.650107,-8.909095,-0.258988
3,2023-08-15,2023-11-30,C,4180.0,0.293151,0.185528,343.818889,343.822289,0.003400,0.754825,...,-0.000474,-298.781901,-298.808496,-0.026595,754.528321,755.666172,1.137851,879.752688,881.060081,1.307393
4,2023-08-15,2024-08-16,C,3450.0,1.005479,0.258598,1218.340257,1218.468873,0.128616,0.896422,...,-0.002996,-162.209325,-160.815860,1.393465,818.680291,809.591830,-9.088461,2862.180021,2863.683476,1.503455
